In [0]:
# =====================================================================
# reversion / 01_reversion.py
# Ejecuta el DROP lógico (reversion/01_reversion.sql) y borra las rutas
# físicas correspondientes en exlt-retail-data, ya que las tablas son
# EXTERNAL con LOCATION explícito (mismo patrón que en clase).
# =====================================================================

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("sql_file", "01_reversion.sql")
dbutils.widgets.text("catalogo", "retail_medallion")
dbutils.widgets.text("storageName", "adlsretailproject0826")

sql_file = dbutils.widgets.get("sql_file")
catalogo = dbutils.widgets.get("catalogo")
storageName = dbutils.widgets.get("storageName")

In [0]:
with open(sql_file, "r", encoding="utf-8") as f:
    script = f.read()

script = script.replace("${catalogo}", catalogo)

for stmt in [s.strip() for s in script.split(";") if s.strip() and not s.strip().startswith("--")]:
    try:
        spark.sql(stmt)
        print(f"OK -> {stmt[:80]}")
    except Exception as e:
        print(f"SKIP -> {stmt[:80]} ({e})")

OK -> USE CATALOG retail_medallion
OK -> DROP TABLE IF EXISTS retail_medallion.golden.fact_sales
OK -> DROP TABLE IF EXISTS retail_medallion.golden.dim_product
OK -> DROP TABLE IF EXISTS retail_medallion.golden.dim_customer
OK -> DROP TABLE IF EXISTS retail_medallion.golden.agg_sales_by_category_month
OK -> DROP TABLE IF EXISTS retail_medallion.golden.agg_sales_by_holiday
OK -> DROP TABLE IF EXISTS retail_medallion.golden.ecommerce_marketing_roi
OK -> DROP TABLE IF EXISTS retail_medallion.silver.superstore_clean
OK -> DROP TABLE IF EXISTS retail_medallion.silver.ecommerce_daily_clean
OK -> DROP TABLE IF EXISTS retail_medallion.bronze.superstore_raw
OK -> DROP TABLE IF EXISTS retail_medallion.bronze.ecommerce_raw
OK -> DROP TABLE IF EXISTS retail_medallion.bronze.calendar_holidays


In [0]:
# ### Eliminación de rutas físicas (tablas EXTERNAL)
base = f"abfss://retail-data@{storageName}.dfs.core.windows.net"
paths = [
    f"{base}/bronze/superstore_raw",
    f"{base}/bronze/ecommerce_raw",
    f"{base}/bronze/calendar_holidays",
    f"{base}/silver/superstore_clean",
    f"{base}/silver/ecommerce_daily_clean",
    f"{base}/golden/dim_product",
    f"{base}/golden/dim_customer",
    f"{base}/golden/fact_sales",
    f"{base}/golden/agg_sales_by_category_month",
    f"{base}/golden/agg_sales_by_holiday",
    f"{base}/golden/ecommerce_marketing_roi",
]
for path in paths:
    dbutils.fs.rm(path, recurse=True)
    print(f"Ruta física eliminada: {path}")

Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/bronze/superstore_raw
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/bronze/ecommerce_raw
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/bronze/calendar_holidays
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/silver/superstore_clean
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/silver/ecommerce_daily_clean
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/golden/dim_product
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/golden/dim_customer
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/golden/fact_sales
Ruta física eliminada: abfss://retail-data@adlsretailproject0826.dfs.core.windows.net/golden/agg_sales_by_category_month
Ruta física e